# E1 · W9 · Kaggle run 18 — benchmark e1pipe từng frame (GPU T4)
**Mục đích (W9-01 #129, W9-02 #130; `docs/w9/W9_PLAN.md` mục B, C):** chạy pipeline từng frame `e1pipe` (decode → detect → track → R1 → ghi) trên 38 video **val** cố định, ba cấu hình × 3 lần, đan xen để tránh trôi nhiệt độ/bộ nhớ đệm:
- C0 PyTorch fp32 · C1 PyTorch fp16 · C2 TensorRT fp16 (nếu xuất engine thất bại thì ghi FAILED, không thử cách khác);
- lần 1 của mỗi cấu hình lưu track/det/sự kiện để kiểm độ chính xác và cổng tái tạo (so với run 17);
- cuối cùng CPU fp32 trên 5 video đầu (chỉ báo cáo).

Luật áp dụng ngoài Kaggle (`scripts/experiments/w9_bench_select.py`). Test vẫn khóa. **Trước khi chạy:** xem [LOCAL] `docs/w9/kaggle_guide_w9_LOCAL.md`.


In [ ]:
COMMIT = ""                      # <-- SHA day du 40 ky tu (cung commit voi dataset e1-w9-tools)
WEIGHTS_SHA = "a3de1e0195e4380aac497612296ed57f67e41ceb5ae3314e82a5ad66b125ef8a"  # best.pt yolo11n_640_s20261030 (run 13)
ULTRALYTICS_VERSION = "8.4.183"
RUNNER = "manual"                # "github-actions" khi chay qua workflow (ADR-0016)
PLAN = [("C0", 1), ("C1", 1), ("C2", 1), ("C0", 2), ("C1", 2), ("C2", 2), ("C0", 3), ("C1", 3), ("C2", 3)]


In [ ]:
import glob, hashlib, json, os, subprocess, sys, zipfile
from pathlib import Path

def find(name):
    hits = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    assert hits, f"KHONG TIM THAY {name} trong /kaggle/input — kiem tra Add Input"
    if len(hits) > 1:
        print(f"[canh bao] {name} co {len(hits)} ban, dung ban dau: {hits}")
    return hits[0]

def sha(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for c in iter(lambda: f.read(1 << 20), b""):
            h.update(c)
    return h.hexdigest()

assert len(COMMIT) == 40 and all(c in "0123456789abcdef" for c in COMMIT.lower()), \
    "Dien COMMIT = SHA day du 40 ky tu (xem huong dan LOCAL)"
if not glob.glob("/kaggle/input/**/TOOLS_MANIFEST.json", recursive=True):
    zipfile.ZipFile(find("e1-w9-tools.zip")).extractall("/kaggle/working/tools")
TOOLS = str(Path((glob.glob("/kaggle/input/**/TOOLS_MANIFEST.json", recursive=True) or glob.glob("/kaggle/working/tools/**/TOOLS_MANIFEST.json", recursive=True))[0]).parent)
tm = json.load(open(f"{TOOLS}/TOOLS_MANIFEST.json"))
assert tm["commit"] == COMMIT, f"dataset tools dung commit {tm['commit'][:12]}, khac COMMIT — tao lai tools bang make_tools_bundle.py"
assert not tm["dirty_worktree"], "tools dung tu thu muc co thay doi chua commit"
bad = [f["path"] for f in tm["files"] if sha(f"{TOOLS}/{f['path']}") != f["sha256"]]
assert not bad, f"tep tools lech SHA: {bad[:5]}"
W = [p for p in sorted(glob.glob("/kaggle/input/**/best.pt", recursive=True)) if sha(p) == WEIGHTS_SHA]
assert W, "Khong thay best.pt co SHA-256 cua yolo11n_640_s20261030 — kiem tra input e1-w5-run13"
WEIGHTS = W[0]
print("TOOLS", TOOLS, "| commit", tm["commit"][:12], "|", len(tm["files"]), "tep | WEIGHTS", WEIGHTS)


In [ ]:
!pip -q install "lap>=0.5.12" {"ultralytics==" + ULTRALYTICS_VERSION}
import lap, torch, ultralytics
print("ultralytics", ultralytics.__version__, "| lap", lap.__version__, "| cuda", torch.cuda.is_available())
assert ultralytics.__version__ == ULTRALYTICS_VERSION
assert torch.cuda.is_available(), "Chua bat GPU: Settings -> Accelerator -> GPU T4 x2"


In [ ]:
# Du lieu tho Svanstrom: neu dataset la .zip thi giai nen vao /kaggle/working/raw
if not glob.glob("/kaggle/input/**/Data/Video_V", recursive=True) and not glob.glob("/kaggle/working/raw/**/Data/Video_V", recursive=True):
    for z in glob.glob("/kaggle/input/**/*.zip", recursive=True):
        if "svanstrom" in z.lower() or "drone-detection" in z.lower():
            print("giai nen", z); zipfile.ZipFile(z).extractall("/kaggle/working/raw")
DATA_ROOT = "/kaggle/working/raw" if glob.glob("/kaggle/working/raw/**/Data/Video_V", recursive=True) else "/kaggle/input"
assert glob.glob(f"{DATA_ROOT}/**/Data/Video_V", recursive=True), "Khong thay Data/Video_V — kiem tra input e1-svanstrom-raw"
BENCH = f"{TOOLS}/scripts/kaggle/w9_e1pipe_bench.py"
print("DATA_ROOT =", DATA_ROOT)


## 1. Ba cấu hình × 3 lần (mỗi lần một tiến trình mới)

In [ ]:
RC = {}
for cfg, rep in PLAN:
    extra = ["--save-outputs"] if rep == 1 else []
    cmd = [sys.executable, BENCH, "--data-root", DATA_ROOT, "--tools", TOOLS, "--weights", WEIGHTS, "--weights-sha", WEIGHTS_SHA,
           "--config", cfg, "--rep", str(rep), "--out", "/kaggle/working", "--commit", COMMIT, "--runner", RUNNER, *extra]
    print(">>>", cfg, rep, flush=True)
    RC[(cfg, rep)] = subprocess.run(cmd).returncode
    print("<<<", cfg, rep, "ma thoat", RC[(cfg, rep)], flush=True)
print(RC)


## 2. CPU fp32, 5 video đầu (chỉ báo cáo)

In [ ]:
RC[("CPU", 1)] = subprocess.run([sys.executable, BENCH, "--data-root", DATA_ROOT, "--tools", TOOLS, "--weights", WEIGHTS,
    "--weights-sha", WEIGHTS_SHA, "--config", "CPU", "--rep", "1", "--max-videos", "5", "--out", "/kaggle/working",
    "--commit", COMMIT, "--runner", RUNNER]).returncode
print(RC)


## Tự kiểm tra (phải in `PASS`)

In [ ]:
import pandas as pd
rows = []
for (cfg, rep), rc in RC.items():
    d = f"/kaggle/working/w9_{cfg}_rep{rep}"
    st = json.load(open(f"{d}/stage_status.json")) if os.path.exists(f"{d}/stage_status.json") else {}
    s = json.load(open(f"{d}/summary.json")) if os.path.exists(f"{d}/summary.json") else {}
    rows.append({"cfg": cfg, "rep": rep, "rc": rc, "stages": ",".join(f"{k}:{v['status']}" for k, v in st.items()),
                 "frames": s.get("frames"), "fps": s.get("fps_excl_warmup"), "p95_ms": (s.get("latency_ms") or {}).get("total", {}).get("p95")})
df = pd.DataFrame(rows); print(df.to_string())
ok_c0 = all(r["rc"] == 0 and r["frames"] == 11821 for r in rows if r["cfg"] in ("C0", "C1"))
c2 = [r for r in rows if r["cfg"] == "C2"]
ok_c2 = all(r["rc"] == 0 and r["frames"] == 11821 for r in c2) or all(r["rc"] == 2 for r in c2)
print("C2:", "OK" if all(r["rc"] == 0 for r in c2) else ("XUAT ENGINE THAT BAI (hop le theo luat, ghi lai)" if all(r["rc"] == 2 for r in c2) else "LOI"))
print("PASS" if ok_c0 and ok_c2 else "FAIL - gui cac stage_status.json cho Claude, KHONG chay lai")
zs = sorted(glob.glob("/kaggle/working/w9_*_evidence.zip"))
with zipfile.ZipFile("/kaggle/working/w9_run18_all.zip", "w", zipfile.ZIP_STORED) as zf:
    for z in zs:
        zf.write(z, os.path.basename(z))
print("w9_run18_all.zip", os.path.getsize("/kaggle/working/w9_run18_all.zip") // 1024, "KB  sha256", sha("/kaggle/working/w9_run18_all.zip"))
